In [1]:
import torch
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
import torch.nn.functional as F

# 1. Setup the Graph (NY Taxi Style)
# Let's assume 100 Zones in NYC
num_zones = 100
x = torch.randn((num_zones, 16)) # 16 features per zone (e.g., avg speed, weather)

# 2. Create Edges based on trips
# edge_index: [Source Zone, Target Zone]
# edge_attr:  [Number of trips] 
edge_index = torch.randint(0, num_zones, (2, 500)) 
edge_weight = torch.rand(500) # Weights representing trip intensity (0.0 to 1.0)

data = Data(x=x, edge_index=edge_index, edge_attr=edge_weight)

# 3. GNN Model for Flow Prediction
class TransportGNN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        # Note: GCNConv can take 'edge_weight' into account!
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, 1) # Outputting a single value per node

    def forward(self, x, edge_index, edge_weight):
        # We pass the trip intensity (edge_weight) into the convolution
        x = self.conv1(x, edge_index, edge_weight)
        x = F.relu(x)
        x = self.conv2(x, edge_index, edge_weight)
        return x

# Initialize
model = TransportGNN(in_channels=16, hidden_channels=32)
out = model(data.x, data.edge_index, data.edge_attr)

print(f"Prediction for each zone: {out.shape}") # [100, 1]

/home/tom_duq/anaconda3/envs/myenv/lib/python3.12/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: Could not load this library: /home/tom_duq/anaconda3/envs/myenv/lib/python3.12/site-packages/torch_scatter/_version_cpu.so
  import torch_geometric.typing
/home/tom_duq/anaconda3/envs/myenv/lib/python3.12/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: Could not load this library: /home/tom_duq/anaconda3/envs/myenv/lib/python3.12/site-packages/torch_cluster/_version_cpu.so
  import torch_geometric.typing
/home/tom_duq/anaconda3/envs/myenv/lib/python3.12/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-spline-conv'. Disabling its usage. Stacktrace: Could not load this library: /home/tom_duq/anaconda3/envs/myenv/lib/python3.12/site-packages/torch_spline_conv

Prediction for each zone: torch.Size([100, 1])


In [2]:
#!pip install pyarrow

import pandas as pd
import torch
import torch.nn.functional as F
from torch_geometric.data import Data

# 1. Load the data using the newly installed pyarrow engine
df = pd.read_parquet('yellow_tripdata_2025-08.parquet', engine='pyarrow')

# 2. Filter for valid NYC Taxi Zones (1 to 263)
df = df[(df['PULocationID'] <= 263) & (df['DOLocationID'] <= 263)]

# 3. Create Weighted Edges (Trip Counts)
# We group by Pickup and Dropoff to find how many trips happened between zones
adj_df = df.groupby(['PULocationID', 'DOLocationID']).size().reset_index(name='trip_count')

# Convert to 0-indexed for PyTorch (Zone 1 becomes Index 0)
edge_index = torch.tensor([
    adj_df['PULocationID'].values - 1,
    adj_df['DOLocationID'].values - 1
], dtype=torch.long)

# Normalize trip counts to use as edge weights
edge_weight = torch.tensor(adj_df['trip_count'].values, dtype=torch.float)
edge_weight = edge_weight / edge_weight.max() 

# 4. Create Node Features (Average trip distance/fare per zone)
zone_stats = df.groupby('PULocationID').agg({
    'trip_distance': 'mean',
    'fare_amount': 'mean'
}).reindex(range(1, 264), fill_value=0)

x = torch.tensor(zone_stats.values, dtype=torch.float)

# 5. Final PyG Data Object
data = Data(x=x, edge_index=edge_index, edge_attr=edge_weight)
print(f"Graph Ready: {data.num_nodes} nodes, {data.num_edges} weighted edges.")

Graph Ready: 263 nodes, 39869 weighted edges.


In [3]:
# Create the target: total trip count per pickup location (Demand)
demand = df.groupby('PULocationID').size().reindex(range(1, 264), fill_value=0)
data.y = torch.tensor(demand.values, dtype=torch.float).view(-1, 1)

# Normalize the target for better training convergence
target_mean = data.y.mean()
target_std = data.y.std()
data.y = (data.y - target_mean) / target_std

from torch_geometric.nn import GCNConv

class TaxiRegressionGNN(torch.nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        # Use GCNConv to leverage the trip intensity weights
        self.conv1 = GCNConv(in_channels, 32)
        self.conv2 = GCNConv(32, 16)
        self.out = torch.nn.Linear(16, 1)

    def forward(self, data):
        x, edge_index, edge_weight = data.x, data.edge_index, data.edge_attr
        
        # Message passing scales information by the edge weight
        x = F.relu(self.conv1(x, edge_index, edge_weight))
        x = F.relu(self.conv2(x, edge_index, edge_weight))
        
        return self.out(x)

model = TaxiRegressionGNN(in_channels=data.num_features)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = torch.nn.MSELoss() # Mean Squared Error for regression

model.train()
for epoch in range(200):
    optimizer.zero_grad()
    out = model(data)
    loss = criterion(out, data.y)
    loss.backward()
    optimizer.step()
    
    if epoch % 20 == 0:
        print(f'Epoch {epoch:03d}, Loss: {loss.item():.4f}')

print("Training Complete!")

Epoch 000, Loss: 1.0746
Epoch 020, Loss: 0.7548
Epoch 040, Loss: 0.5789
Epoch 060, Loss: 0.3819
Epoch 080, Loss: 0.3288
Epoch 100, Loss: 0.2885
Epoch 120, Loss: 0.2455
Epoch 140, Loss: 0.2206
Epoch 160, Loss: 0.1982
Epoch 180, Loss: 0.1873
Training Complete!


In [4]:
from sklearn.metrics import mean_absolute_error

model.eval()
with torch.no_grad():
    y_pred = model(data)
    y_pred_unnorm = (y_pred * target_std) + target_mean
    y_true_unnorm = (data.y * target_std) + target_mean
    
    mae = mean_absolute_error(y_true_unnorm.cpu(), y_pred_unnorm.cpu())
    print(f"Average Prediction Error: {mae:.2f} trips per zone")

Average Prediction Error: 5374.74 trips per zone


In [11]:
from torch_geometric.nn import GATConv

class TaxiGAT(torch.nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        # 8 attention heads allow the model to learn 8 different types of 
        # spatial relationships simultaneously
        self.gat1 = GATConv(in_channels, 32, heads=8, concat=True, edge_dim=1)
        self.gat2 = GATConv(32 * 8, 16, heads=1, concat=False, edge_dim=1)
        self.out = torch.nn.Linear(16, 1)

    def forward(self, data):
        x, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr
        
        # We pass edge_attr (trip counts) so the attention mechanism 
        # can use the physical flow as a hint
        x = F.elu(self.gat1(x, edge_index, edge_attr.view(-1, 1)))
        x = F.elu(self.gat2(x, edge_index, edge_attr.view(-1, 1)))
        
        return self.out(x)

# Initialize new model
model = TaxiGAT(in_channels=data.num_features)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005) # Lower LR for GAT stability

# Calculate absolute error per zone
residuals = torch.abs(y_true_unnorm - y_pred_unnorm).cpu().numpy()
top_error_zones = residuals.argsort()[-5:][::-1]

print("Zones with the highest error (likely hubs or anomalies):")
for zone_idx in top_error_zones:
    print(f"Zone {zone_idx + 1}: Error of {residuals[zone_idx]} trips")

Zones with the highest error (likely hubs or anomalies):
Zone [1]: Error of [[6232.758]] trips
Zone [1]: Error of [[6232.758]] trips
Zone [1]: Error of [[6232.758]] trips
Zone [1]: Error of [[6232.758]] trips
Zone [1]: Error of [[6232.758]] trips


In [13]:
from pyvis.network import Network
import numpy as np

def visualize_taxi_graph(data, num_nodes=100):
    net = Network(height='750px', width='100%', bgcolor='#000000', font_color='white')
    
    # 1. Prepare Node sizes
    # Un-normalize demand and cast to standard Python float
    node_size_base = data.y.flatten().detach().cpu().numpy()
    node_size_norm = (node_size_base - node_size_base.min()) / (node_size_base.max() - node_size_base.min() + 1e-6)

    for i in range(min(num_nodes, data.num_nodes)):
        # Use float() to ensure JSON serializability
        size = float(10 + (node_size_norm[i] * 50))  
        avg_dist = float(data.x[i, 0].item())
        
        net.add_node(i, 
                     label=f"Zone {i+1}", 
                     title=f"Avg Distance: {avg_dist:.2f} miles",
                     size=size,
                     color='#00ffcc')

    # 2. Prepare Edges
    edge_index = data.edge_index.cpu().numpy()
    edge_weights = data.edge_attr.cpu().numpy()
    
    mask = (edge_index[0] < num_nodes) & (edge_index[1] < num_nodes)
    filtered_edges = edge_index[:, mask]
    filtered_weights = edge_weights[mask]

    for i in range(filtered_edges.shape[1]):
        u, v = int(filtered_edges[0, i]), int(filtered_edges[1, i])
        # Use float() here to fix the JSON error
        width = float(filtered_weights[i] * 15) 
        
        if width > 0.5:
            net.add_edge(u, v, width=width, color='#555555', alpha=0.3)

    net.toggle_physics(True)
    # Using write_html to avoid the browser-opening error on headless servers
    net.write_html('nyc_taxi_flow.html')
    print("Success! Download 'nyc_taxi_flow.html' to explore the corridors.")

visualize_taxi_graph(data, num_nodes=150)


Success! Download 'nyc_taxi_flow.html' to explore the corridors.


In [15]:
from sklearn.cluster import KMeans
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from pyvis.network import Network
import torch.nn.functional as F

def visualize_colored_clusters(model, data, num_clusters=7, num_nodes=150):
    model.eval()
    with torch.no_grad():
        # 1. Extract the embeddings (z) using the correct layer names: gat1 and gat2
        # We process the features through the GAT layers to get the learned representations
        x, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr
        edge_attr = edge_attr.view(-1, 1)
        
        # Changed model.conv1 -> model.gat1 and model.conv2 -> model.gat2
        z = F.elu(model.gat1(x, edge_index, edge_attr))
        z = F.elu(model.gat2(z, edge_index, edge_attr)).cpu().numpy()

    # 2. Cluster these embeddings into 7 groups to match your t-SNE plot clusters
    kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
    labels = kmeans.fit_predict(z)

    # 3. Create a color map matching the t-SNE palette ('Set2')
    cmap = cm.get_cmap('Set2', num_clusters)
    cluster_colors = [mcolors.to_hex(cmap(i)) for i in range(num_clusters)]

    # 4. Build the Interactive Map with black background for better contrast
    net = Network(height='750px', width='100%', bgcolor='#111111', font_color='white')
    
    # Scale node size by actual trip demand per zone
    node_sizes = data.y.flatten().cpu().numpy()
    node_sizes_norm = (node_sizes - node_sizes.min()) / (node_sizes.max() - node_sizes.min() + 1e-6)

    for i in range(min(num_nodes, data.num_nodes)):
        cluster_id = int(labels[i])
        net.add_node(i, 
                     label=f"Zone {i+1}", 
                     title=f"Cluster: {cluster_id}", 
                     color=cluster_colors[cluster_id],
                     size=float(10 + (node_sizes_norm[i] * 50)))

    # 5. Add weighted edges representing taxi trip flow
    edge_idx = data.edge_index.cpu().numpy()
    edge_w = data.edge_attr.cpu().numpy()
    for i in range(len(edge_w)):
        u, v = int(edge_idx[0, i]), int(edge_idx[1, i])
        # Filter for readability and only draw edges within our node subset
        if u < num_nodes and v < num_nodes and edge_w[i] > 0.05:
            net.add_edge(u, v, width=float(edge_w[i] * 10), color='#444444', alpha=0.2)

    net.write_html('nyc_taxi_colored_clusters.html')
    print("Success! Created 'nyc_taxi_colored_clusters.html'.")

# Run the fixed visualization
visualize_colored_clusters(model, data)

Success! Created 'nyc_taxi_colored_clusters.html'.


/tmp/ipykernel_10650/2472369620.py:24: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = cm.get_cmap('Set2', num_clusters)
